# Rakentaminen Meta-perhemalleilla 

## Johdanto 

Tässä oppitunnissa käsitellään: 

- Kahden pääasiallisen Meta-perhemallin tutkiminen - Llama 3.1 ja Llama 3.2, sekä koodiesimerkeissä käytetyt tällä hetkellä saatavilla olevat vaihtoehdot (Llama 3.3 ja Llama 4) 
- Jokaisen mallin käyttötarkoitusten ja tilannekuvien ymmärtäminen 
- Koodiesimerkki, joka esittelee kunkin mallin ainutlaatuiset ominaisuudet 


## Meta-perhemallit 

Tässä oppitunnissa tutustumme kahteen Meta-perheen tai ”Llama Herdin” malliin - Llama 3.1 ja Llama 3.2. Suorituskelpoiset koodiesimerkit käyttävät Llama 3.3:ta ja Llama 4:ää, jotka ovat tällä hetkellä saatavilla olevia vaihtoehtoja Microsoft Foundry Models -katalogissa. 

Nämä mallit ovat eri variantteja ja ne ovat saatavilla [Microsoft Foundry Models -katalogissa](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst).

> **Huom:** GitHub Models lopetetaan heinäkuun 2026 lopussa. Tässä on lisätietoja [Microsoft Foundry Modelsin](https://learn.microsoft.com/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) käyttämisestä AI-mallien prototyyppien tekemiseen.

> **Huom:** Tässä oppitunnissa alun perin käytetyt malliversiot (Llama 3.1 70B/405B Instruct, Llama 3.2 11B/90B Vision Instruct) on sittemmin poistettu Microsoft Foundry Models -katalogista. Alla olevissa koodiesimerkeissä käytetään `Llama-3.3-70B-Instruct` ja `Llama-4-Scout-17B-16E-Instruct`, jotka ovat tällä hetkellä saatavilla olevia vaihtoehtoja Microsoft Foundry Models -katalogissa, joten ne toimivat edelleen tänään.

Mallivariantit (kuten alun perin tässä oppitunnissa käsitelty): 
- Llama 3.1 - 70B Instruct (poistettu) 
- Llama 3.1 - 405B Instruct (poistettu) 
- Llama 3.2 - 11B Vision Instruct (poistettu) 
- Llama 3.2 - 90B Vision Instruct (poistettu) 

Tällä hetkellä saatavilla olevat vaihtoehdot, joita käytetään koodiesimerkeissä: 
- Llama 3.3 - 70B Instruct (`Llama-3.3-70B-Instruct`) 
- Llama 4 Scout - 17B 16E Instruct (`Llama-4-Scout-17B-16E-Instruct`) 

*Huomautus: Yllä käsitellyt Llama 3 -variantit on säilytetty historiallisessa kontekstissa eivätkä ne ole enää saatavilla Microsoft Foundry Modelsissa.*



## Llama 3.1

405 miljardin parametrin kokoluokassaan Llama 3.1 kuuluu avoimen lähdekoodin LLM-kategoriaan.

Tämä versio on päivitys aiempaan Llama 3 -julkaisuun ja tarjoaa:

- Suuremman kontekstikehyksen – 128k tokenia vs. 8k tokenia
- Suuremman maksimivastauksen pituuden – 4096 vs. 2048 tokenia
- Parempi monikielinen tuki – lisättyjen koulutustokenien ansiosta

Näiden ansiosta Llama 3.1 pystyy käsittelemään monimutkaisempia käyttötapauksia GenAI-sovelluksia rakentaessa, mukaan lukien:
- Natiivinen funktiokutsuminen – kyky kutsua ulkoisia työkaluja ja funktioita LLM-työnkulun ulkopuolella
- Parempi RAG-suorituskyky – suuremman kontekstikehyksen ansiosta
- Syntetisoidun datan generointi – kyky luoda tehokasta dataa esimerkiksi hienosäätöä varten



### Natiivifunktioiden kutsuminen 

Llama 3.1:ää on hienosäädetty toimimaan tehokkaammin funktio- tai työkalukutsujen tekemisessä. Siinä on myös kaksi sisäänrakennettua työkalua, jotka malli voi tunnistaa tarpeellisiksi käyttää käyttäjän kehotteen perusteella. Nämä työkalut ovat: 

- **Brave Search** - Voidaan käyttää ajantasaisen tiedon, kuten säätietojen, saamiseen tekemällä verkkohaku 
- **Wolfram Alpha** - Voidaan käyttää monimutkaisempiin matemaattisiin laskelmiin, joten omien funktioiden kirjoittaminen ei ole tarpeen. 

Voit myös luoda omia räätälöityjä työkaluja, joita LLM voi kutsua. 

Seuraavassa koodiesimerkissä: 

- Määrittelemme käytettävissä olevat työkalut (brave_search, wolfram_alpha) järjestelmäkehotteessa. 
- Lähetä käyttäjän kehotus, joka kysyy tietyn kaupungin säästä. 
- LLM vastaa työkalukutsulla Brave Search -työkaluun, joka näyttää tältä `<|python_tag|>brave_search.call(query="Stockholm weather")` 

*Huomautus: Tämä esimerkki tekee vain työkalukutsun, jos haluat saada tulokset, sinun on luotava ilmainen tili Brave API:n sivulla ja määritettävä itse funktio` 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Meta-Llama-3.1-405B-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-3.3-70B-Instruct, a currently available alternative, instead
model_name = "Llama-3.3-70B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

Vaikka Llama 3.1 on suuri kielimalli (LLM), yksi sen rajoituksista on multimodaalisuus. Eli kyky käyttää erilaisia syötetyyppejä, kuten kuvia kehotteina ja vastauksina. Tämä kyky on yksi Llama 3.2:n pääominaisuuksista. Nämä ominaisuudet sisältävät myös: 

- Multimodaalisuus - kykenee käsittelemään sekä teksti- että kuvatulosteita 
- Pienet ja keskisuuret kokovaihtoehdot (11B ja 90B) - tämä tarjoaa joustavia käyttöönottoasetuksia, 
- Vain teksti -vaihtoehdot (1B ja 3B) - tämä mahdollistaa mallin käytön reunalla / mobiililaitteissa ja tarjoaa matalan viiveen 

Multimodaalituki on suuri askel avoimen lähdekoodin mallien maailmassa. Alla oleva koodiesimerkki ottaa sekä kuvan että tekstikehotteen analysoidakseen kuvaa. Koska Llama 3.2 90B Vision Instruct on poistettu käytöstä, esimerkissä käytetään `Llama-4-Scout-17B-16E-Instruct`, tällä hetkellä saatavilla olevaa natiivia multimodaalista vaihtoehtoa. 

### Multimodaalituki Llama 3.2:n kanssa


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Llama-3.2-90B-Vision-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-4-Scout-17B-16E-Instruct, a currently available natively multimodal alternative, instead
model_name = "Llama-4-Scout-17B-16E-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## Oppiminen ei lopu tähän, jatka matkaa

Tämän oppitunnin jälkeen tutustu [Generative AI Learning -kokoelmaamme](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst) jatkaaksesi Generative AI -taitojesi kehittämistä!


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
